In [0]:
catalog = "databricks_virtue_foundation_dataset_dais_2026"
schema = "virtue_foundation_dataset"

pincode_clean = spark.sql(f"""
    WITH base AS (
        SELECT
            CAST(pincode AS STRING)         AS pincode_str,
            UPPER(TRIM(district))           AS district_clean,
            UPPER(TRIM(statename))          AS state_clean,
            COUNT(*)                        AS office_count
        FROM {catalog}.{schema}.india_post_pincode_directory
        WHERE
            pincode IS NOT NULL
            AND LENGTH(CAST(pincode AS STRING)) = 6
            AND district IS NOT NULL
            AND TRIM(district) != ''
            AND TRIM(district) != 'NA'
            AND statename IS NOT NULL
            AND TRIM(statename) != ''
            AND TRIM(statename) != 'NA'
        GROUP BY pincode, district, statename
    )
    SELECT
        pincode_str,
        district_clean,
        state_clean,
        office_count
    FROM base
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY pincode_str
        ORDER BY office_count DESC
    ) = 1
""")

pincode_clean.createOrReplaceTempView("pincode_deduped")

# sanity checks
total_raw = spark.sql(f"""
    SELECT COUNT(*) as raw_rows,
           COUNT(DISTINCT pincode) as unique_pincodes
    FROM {catalog}.{schema}.india_post_pincode_directory
""").collect()[0]

total_clean = pincode_clean.count()

print(f"raw rows:          {total_raw['raw_rows']}")
print(f"unique pincodes:   {total_raw['unique_pincodes']}")
print(f"after dedup:       {total_clean}")
print(f"pincodes lost:     {total_raw['unique_pincodes'] - total_clean}")
print(f"temp view saved:   pincode_deduped")

# spot check
spark.sql("SELECT * FROM pincode_deduped LIMIT 5").display()